# Volume 3, Chapter 7: Random Walks (Python companion)

Exercises first, solutions at the end. We watch the arcsine laws appear, play roulette with the gambler's ruin formula, estimate Pólya's return probability in three dimensions in two different ways (by summing a series and by measuring an electrical resistance), solve the discrete Dirichlet problem, and run Papadimitriou's random walk for 2-SAT and Schöning's for 3-SAT.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
Phi = lambda t: 0.5 * (1 + math.erf(t / math.sqrt(2)))
def u(k):
    """P(S_2k = 0) for the simple walk on Z, computed in logs."""
    return math.exp(math.lgamma(2*k + 1) - 2*math.lgamma(k + 1) - 2*k*math.log(2))

## Exercises

**P7.1 (★) The two arcsine laws.** Simulate $20\,000$ walks of $2n=1000$ steps. For each, record the time $L_{2n}$ of the last zero and the number $P_{2n}$ of positive steps. Compare both histograms with $u_{2k}u_{2n-2k}$ (Theorem 7.6 and Exercise 7.3) and with the arcsine density. Also check Theorem 7.3(a) by simulation.

**P7.2 (★) Roulette.** Check Theorem 7.9 by simulation for $p=\frac{18}{37}$, $x=10$, $N=20$ (success probability and mean duration). Then tabulate the success probability from $100$ euros with target $200$ for stakes of $1,2,5,10,20,50,100$ euros.

**P7.3 (★★) Pólya's constant by a series.** In $\Z^3$, $u_{2n}=\binom{2n}n4^{-n}\sum_{i+j+k=n}\bigl(\frac{n!}{i!\,j!\,k!}\bigr)^29^{-n}$ (choose how many of the $2n$ steps go along each axis, and return on each). Compute $\sum_{n\le N}u_{2n}$ for $N$ up to $2000$, deduce estimates of the return probability $F=1-1/\sum_nu_n$ (Theorem 7.12), and correct for the tail using $u_{2n}\sim2\bigl(\frac3{4\pi n}\bigr)^{3/2}$. Compare with simulated walks in dimensions $1$, $2$, $3$ that are stopped after $10^4$ steps.

**P7.4 (★★) Pólya's constant by a resistance.** By Exercise 7.8, the walk from $0$ reaches the boundary of the box $[-L,L]^d$ before returning to $0$ with probability $\frac1{2d\,R_L}$, where $R_L$ is the effective resistance between $0$ and the boundary. Compute $R_L$ for $d=2$ and $d=3$ by solving the discrete Dirichlet problem with the conjugate gradient method, for $L$ up to $40$. Watch $R_L\to\infty$ in the plane (slowly) and $R_L\to\frac1{6(1-F)}$ in space.

**P7.5 (★) The Dirichlet problem.** Solve the problem of Figure 7.4 (L-shaped domain, $g=1$ on the top edge) by iteration, and compare $f(4,8)$ with a Monte Carlo estimate from $10^5$ walks.

**P7.6 (★★) Papadimitriou's 2-SAT walk.** Generate random satisfiable 2-SAT formulas with $n$ variables and $3n$ clauses (plant a hidden satisfying assignment and keep only clauses it satisfies). Run the walk of Theorem 7.15 and plot the mean number of flips against $n$. Then build a formula on which the walk is slow: how close to $n^2$ can you get?

**P7.7 (★★) Schöning's 3-SAT walk.** For planted 3-SAT formulas with $n$ variables and $5n$ clauses, repeat: pick a random assignment and run the walk for $3n$ flips. Count the restarts needed for $n=10,\dots,40$, and compare the growth with $(4/3)^n$.

**P7.8 (★) The maximum.** Using Proposition 7.5, compute the exact law of $M_n$ for $n=10^4$ and compare $P(M_n\ge t\sqrt n)$ with $2(1-\Phi(t))$ (Exercise 7.2).

## Solutions

**P7.1.** `cumsum` along each row gives the walks; the last zero is found from the reversed boolean array.

In [ ]:
m, n2 = 20000, 1000
S = np.cumsum(rng.choice(np.array([-1, 1], dtype=np.int16), size=(m, n2)), axis=1, dtype=np.int16)
zero = (S == 0)
last = np.where(zero.any(axis=1), n2 - np.argmax(zero[:, ::-1], axis=1), 0)       # time of last zero (0 if none)
prev = np.hstack([np.zeros((m, 1), dtype=np.int16), S[:, :-1]])
positive = ((prev + S) > 0).sum(axis=1)
n = n2 // 2
k = np.arange(n + 1)
exact = np.array([u(j) * u(n - j) for j in k])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, data, title in [(ax[0], last, 'last zero L_2n / 2n'), (ax[1], positive, 'positive steps P_2n / 2n')]:
    a.hist(data / n2, bins=50, density=True, alpha=0.6, label='simulation')
    a.plot(k / n, exact * n, 'k-', lw=1, label='u_2k u_2n-2k (rescaled)')
    a.set_title(title); a.set_ylim(0, 6); a.legend()
plt.show()
print('P(no return by 2n): simulated', np.mean(~zero.any(axis=1)), ' exact u_2n =', u(n))
print('P(L <= 0.1 * 2n): simulated', np.mean(last <= 0.1 * n2), ' arcsine', 2 / math.pi * math.asin(math.sqrt(0.1)))
print('P(more than 90% positive steps): simulated', np.mean(positive >= 0.9 * n2), ' arcsine', 1 - 2 / math.pi * math.asin(math.sqrt(0.9)))

Both histograms follow the same U-shaped curve, the probability of no return by time $1000$ is $u_{1000}\approx0.025$, and a fifth of the games have one player ahead for more than $90\%$ of the time. Note the parity: $P_{2n}$ only takes even values, so the histogram bins must be wide enough to contain several values, or it will show gaps.

**P7.2.**

In [ ]:
def ruin_exact(p, x, N):
    if p == 0.5:
        return x / N, x * (N - x)
    r = (1 - p) / p
    h = (r**x - 1) / (r**N - 1)
    return h, (x - N * h) / (1 - 2 * p)
def ruin_sim(p, x, N, reps=200000):
    pos = np.full(reps, x); t = np.zeros(reps, dtype=np.int64); alive = np.ones(reps, bool)
    while alive.any():
        idx = np.nonzero(alive)[0]
        pos[idx] += np.where(rng.random(len(idx)) < p, 1, -1)
        t[idx] += 1
        alive[idx] = (pos[idx] > 0) & (pos[idx] < N)
    return np.mean(pos == N), t.mean()
p = 18 / 37
print('exact     ', ruin_exact(p, 10, 20))
print('simulated ', ruin_sim(p, 10, 20))
for stake in [1, 2, 5, 10, 20, 50, 100]:
    h, d = ruin_exact(p, 100 // stake, 200 // stake)
    print(f'stake {stake:3d}: P(reach 200) = {h:.4f}   expected number of bets = {d:8.1f}')

The simulation agrees with $h(10)\approx0.368$ and $m(10)\approx98$. The table is the moral of Exercise 7.4: from $0.0045$ with one-euro bets to $0.486$ with a single bet. Each bet hands the house an expected $\frac1{37}$ of the stake, and the expected loss over a whole session is exactly $\frac{1}{37}\times$ (total amount staked), by Wald's identity (Corollary 3.5): fewer, bigger bets stake less in total.

**P7.3.** For each $n$ the inner sum has $O(n^2)$ terms; with logarithms of factorials it is fast enough up to $N=2000$ if we vectorise over $j$.

In [ ]:
lf = np.array([math.lgamma(i + 1) for i in range(4002)])
def u3(n):
    total = 0.0
    for i in range(n + 1):
        j = np.arange(0, n - i + 1)
        kk = n - i - j
        total += np.exp(2 * (lf[n] - lf[i] - lf[j] - lf[kk]) - n * math.log(9)).sum()
    return u(n) * total
N = 2000
us = np.array([u3(n) for n in range(N + 1)])
partial = np.cumsum(us)
for M in [10, 100, 500, 1000, 2000]:
    tail = 2 * (3 / (4 * math.pi))**1.5 * 2 / math.sqrt(M + 0.5)       # sum_{n>M} 2 (3/(4 pi n))^{3/2}
    print(f'N={M:5d}  sum u = {partial[M]:.6f}  F_N = {1 - 1/partial[M]:.5f}   with tail: F = {1 - 1/(partial[M] + tail):.5f}')
print('asymptotic check: u_2n * n^1.5 =', us[N] * N**1.5, ' predicted', 2 * (3 / (4 * math.pi))**1.5)
def returned(d, walks=2000, steps=10000):
    axes = rng.integers(0, d, size=(walks, steps)); signs = rng.choice([-1, 1], size=(walks, steps))
    pos = np.zeros((walks, d), dtype=np.int32); back = np.zeros(walks, bool)
    for t in range(steps):
        pos[np.arange(walks), axes[:, t]] += signs[:, t]
        back |= ~pos.any(axis=1)
    return back.mean()
for d in [1, 2, 3]:
    print(f'd={d}: fraction of walks back at 0 within 10^4 steps: {returned(d):.3f}')

The partial sums converge slowly, since the tail is of order $N^{-1/2}$; with the tail correction the estimate settles at $F\approx0.3405$, Watson's value. The simulation says $\approx0.99$ for $d=1$ (the rest are walks that will return later: $P(T>10^4)=u_{10^4}\approx0.008$), about $0.75$ for $d=2$ (the walk is recurrent, but $P(T>n)$ decays only like $\frac{\pi}{\ln n}$), and about $0.35$ for $d=3$, already close to $F$.

**P7.4.** The voltage $v$ is harmonic in the box minus the origin, with $v(0)=1$ and $v=0$ on the boundary; the current out of $0$ is $I=\sum_{y\sim0}(1-v(y))$ and $R_L=\frac1I$. We solve the linear system for the unknowns inside the box with conjugate gradients, the Laplacian being applied by array slicing.

In [ ]:
def resistance(d, L, tol=1e-10):
    shape = (2 * L + 1,) * d
    c = (L,) * d
    def lap(v):                      # (2d v - sum of neighbours) on the interior, v = 0 on the boundary layer
        w = 2 * d * v
        for ax in range(d):
            w -= np.roll(v, 1, axis=ax) + np.roll(v, -1, axis=ax)
        return w
    mask = np.zeros(shape, bool)
    mask[(slice(1, -1),) * d] = True
    mask[c] = False                  # the origin is fixed at voltage 1
    b = np.zeros(shape); one = np.zeros(shape); one[c] = 1.0
    b[mask] = -lap(one)[mask]        # move the known value v(0)=1 to the right-hand side
    x = np.zeros(shape); r = b.copy(); pdir = r.copy(); rs = (r * r).sum()
    for it in range(20000):
        Ap = lap(pdir); Ap[~mask] = 0
        alpha = rs / (pdir * Ap).sum()
        x += alpha * pdir; r -= alpha * Ap
        rs_new = (r * r).sum()
        if rs_new < tol**2: break
        pdir = r + rs_new / rs * pdir; rs = rs_new
    v = x + one
    I = sum((1 - v[tuple(c[i] + (s if i == ax else 0) for i in range(d))]) for ax in range(d) for s in (1, -1))
    return 1 / I
for L in [2, 5, 10, 20, 40]:
    R2, R3 = resistance(2, L), resistance(3, L)
    print(f'L={L:3d}  d=2: R={R2:.4f}  escape={1/(4*R2):.4f}    d=3: R={R3:.4f}  escape={1/(6*R3):.4f}')
print('1 - F =', 1 - 0.340537, '  so R_infinity =', 1 / (6 * (1 - 0.340537)))

In the plane $R_L$ grows like $\frac1{2\pi}\ln L$ (each doubling of $L$ adds about $0.11=\frac{\ln2}{2\pi}$), so the escape probability $\frac1{4R_L}$ tends to $0$: the walk always comes back. In space $R_L$ converges, to $\approx0.2527$, and the escape probability to $1-F\approx0.6595$. This is Nash-Williams' and Rayleigh's way of thinking: transience means that the lattice, made of unit resistors, conducts current to infinity with finite resistance.

**P7.5.**

In [ ]:
W = H = 16
inD = lambda x, y: 1 <= x <= W - 1 and 1 <= y <= H - 1 and not (x >= 9 and y >= 9)
f = np.zeros((W + 1, H + 1))
f[1:9, H] = 1.0                                      # g = 1 on the top edge
Dmask = np.array([[inD(x, y) for y in range(H + 1)] for x in range(W + 1)])
for _ in range(5000):
    avg = np.zeros_like(f)
    avg[1:-1, 1:-1] = (f[2:, 1:-1] + f[:-2, 1:-1] + f[1:-1, 2:] + f[1:-1, :-2]) / 4
    f = np.where(Dmask, avg, f)
print('f(4,8) by iteration:', f[4, 8])
moves = np.array([(1, 0), (-1, 0), (0, 1), (0, -1)])
reps = 100000
pos = np.tile([4, 8], (reps, 1)); alive = np.ones(reps, bool)
while alive.any():
    idx = np.nonzero(alive)[0]
    pos[idx] += moves[rng.integers(0, 4, len(idx))]
    alive[idx] = Dmask[pos[idx, 0], pos[idx, 1]]
top = (pos[:, 1] == H) & (pos[:, 0] <= 8)
print('Monte Carlo:', top.mean(), '+-', 1.96 * top.std() / math.sqrt(reps))

Both give $f(4,8)\approx0.081$. The iteration (Jacobi's method) is the Dirichlet problem as a computation; the Monte Carlo estimate is Theorem 7.14 as a computation. Monte Carlo is worse here, but it is the only option in high dimension, and it computes one value of $f$ without computing all the others, which is the basis of ``walk on spheres'' methods in computer graphics.

**P7.6.** A literal is a pair (variable, sign). To find a false clause quickly we just scan; this is fine for a few hundred variables.

In [ ]:
def planted(n, m, k, rng):
    hidden = rng.integers(0, 2, n).astype(bool)
    clauses = []
    while len(clauses) < m:
        vs = rng.choice(n, k, replace=False); sg = rng.integers(0, 2, k).astype(bool)
        if any(hidden[v] == s for v, s in zip(vs, sg)):            # the hidden assignment satisfies it
            clauses.append((vs, sg))
    V = np.array([c[0] for c in clauses]); Sg = np.array([c[1] for c in clauses])
    return V, Sg
def walk(V, Sg, n, maxflips, rng, a=None):
    a = rng.integers(0, 2, n).astype(bool) if a is None else a
    for t in range(maxflips + 1):
        false = ~(a[V] == Sg).any(axis=1)
        if not false.any():
            return t, a
        c = np.flatnonzero(false)[0]                               # the first false clause: any rule works
        v = V[c, rng.integers(0, V.shape[1])]
        a[v] = ~a[v]
    return None, a
for n in [25, 50, 100, 200]:
    flips = [walk(*planted(n, 3 * n, 2, rng), n, 10 * n * n, rng)[0] for _ in range(20)]
    print(f'n={n:4d}  mean flips = {np.mean(flips):8.1f}   n^2 = {n*n:6d}   ratio = {np.mean(flips)/n**2:.3f}')
# a slow instance: a chain forcing v_0 = v_1 = ... = v_{n-1}, together with the clause v_0, all variables starting False
def chain(n):
    V = [(0, 0)]; Sg = [(True, True)]
    for i in range(n - 1):
        V.append((i, i + 1)); Sg.append((False, True))             # v_i -> v_{i+1}
        V.append((i, i + 1)); Sg.append((True, False))             # v_{i+1} -> v_i
    return np.array(V), np.array(Sg)
for n in [10, 20, 40]:
    V, Sg = chain(n)
    flips = np.array([walk(V, Sg, n, 50 * n * n, rng, np.zeros(n, bool))[0] for _ in range(400)])
    print(f'chain n={n:4d}  mean flips = {flips.mean():8.1f}   ratio to n^2 = {flips.mean()/n**2:.3f} +- {1.96*flips.std()/math.sqrt(len(flips))/n**2:.3f}')

Random planted formulas are easy: the walk needs far fewer than $n^2$ flips, because most variables are only weakly constrained. The chain $v_0\Leftrightarrow v_1\Leftrightarrow\dots\Leftrightarrow v_{n-1}$ with the clause $(v_0\vee v_0)$ is the worst case: the only satisfying assignment is all true, and from the all-false start the assignment is always of the form $v_0,\dots,v_{k-1}$ true and the rest false, with exactly one false clause. The number $k$ of true variables performs *exactly* the symmetric walk reflected at $0$, so the expected number of flips is exactly $n^2$, and Theorem 7.15 is sharp. (The heavy tail of the flip count, whose standard deviation is about $0.8\,n^2$, is why many runs are needed to see the ratio $1$.) Note that 2-SAT also has a linear-time deterministic algorithm (strongly connected components of the implication graph); the walk's merit is its simplicity and its tiny memory.

**P7.7.**

In [ ]:
def schoening(V, Sg, n, rng):
    restarts = 0
    while True:
        t, a = walk(V, Sg, n, 3 * n, rng)
        if t is not None:
            return restarts
        restarts += 1
ns = [10, 15, 20, 25, 30, 35, 40]
means = []
for n in ns:
    r = [schoening(*planted(n, 5 * n, 3, rng), n, rng) for _ in range(10)]
    means.append(np.mean(r) + 1)
    print(f'n={n:3d}  mean number of tries = {means[-1]:8.1f}   (4/3)^n = {(4/3)**n:9.1f}')
slope = np.polyfit(ns, np.log(means), 1)[0]
print('fitted growth rate per variable:', math.exp(slope), ' (Schoening bound 4/3 =', 4/3, ')')

On planted formulas the walk needs far fewer tries than $(\frac43)^n$: Schöning's bound is a worst case, proved for every satisfiable formula, and planted formulas with many clauses have a strong pull towards the hidden solution. The fitted growth rate is well below $\frac43$, and noisy at these sizes. Hard instances are random formulas near the satisfiability threshold, about $4.27n$ clauses, without a planted solution.

**P7.8.**

In [ ]:
n = 10**4
k = np.arange(n + 1)
logpmf = np.array([math.lgamma(n + 1) - math.lgamma(j + 1) - math.lgamma(n - j + 1) for j in k]) - n * math.log(2)
pS = {2 * j - n: math.exp(lp) for j, lp in zip(k, logpmf)}            # P(S_n = s)
tail = {}                                                              # P(S_n > s)
acc = 0.0
for s in range(n, -n - 1, -1):
    tail[s] = acc; acc += pS.get(s, 0.0)
for t in [0.5, 1, 1.5, 2, 3]:
    r = math.ceil(t * math.sqrt(n))
    pM = 2 * tail[r] + pS.get(r, 0.0)
    print(f't={t}: P(M_n >= t sqrt n) = {pM:.5f}    2(1 - Phi(t)) = {2 * (1 - Phi(t)):.5f}')